# Explainer Notebook — The Hidden Family of Pop
**02805 Social Graphs and Interactions — Final Project**

**Research question:** Is pop music one big family built by a few songwriters, and do artists who share songwriters also sound alike in their lyrics?

Website: TODO link · GitHub: TODO link

In [ ]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, networkx as nx
from popnet import config, build_networks as bn, network_analysis as na, text_analysis as ta, combine
from popnet.cache import load_json

## 1. Motivation
- **What is the data?** TODO: Wikipedia pages of N pop artists + Genius songs, credits and lyrics (2000–2025).
- **Why this data?** TODO: big hits come from a few hidden writers; credits give a natural network, lyrics give text.
- **What should the reader learn?** TODO

## 2. Basic stats
### 2.1 Data collection
TODO: describe the list pages, the infobox filter, Genius matching, the year window. Downloads are cached in `data/raw/`; the code is in `popnet/collect_*.py`.

In [ ]:
# Run once (slow). After that, everything loads from the cache.
# from popnet import collect_wikipedia, collect_genius
# collect_wikipedia.collect_artists(); collect_genius.collect_songs()
artists = load_json(config.DATA_PROCESSED / "artists.json")
songs = load_json(config.DATA_PROCESSED / "songs.json")
matches = pd.read_csv(config.DATA_PROCESSED / "artist_matches.csv")
print(len(artists), "artists |", len(songs), "songs |", matches.genius_id.isna().sum(), "unmatched")

### 2.2 Cleaning and dataset statistics

In [ ]:
df = ta.songs_dataframe(songs)
print("Songs with lyrics:", len(df))
print("Songs with credits:", sum(bool(s["writers"] or s["producers"]) for s in songs))
print("Total tokens:", df.tokens.map(len).sum())
df.groupby("year").size().plot(kind="bar", title="Songs per year"); plt.show()

## 3. Tools, theory and analysis
### 3.1 Building the networks
TODO: explain bipartite artist–creator network, projection, edge weight, threshold `MIN_EDGE_WEIGHT`.

In [ ]:
B = bn.build_bipartite(songs)
G = bn.project_artists(B, min_weight=config.MIN_EDGE_WEIGHT)
pd.DataFrame({"bipartite": na.basic_stats(B), "artist network": na.basic_stats(G)})

### 3.2 Degree distributions — are there a few big hubs? (Q1)

In [ ]:
fig, ax = plt.subplots()
na.plot_degree_distribution([B.degree(n) for n, d in B.nodes(data=True) if d["kind"] == "creator"], "songwriters/producers", ax)
na.plot_degree_distribution([d for _, d in G.degree()], "artists (projection)", ax)
plt.show()
na.top_creators(B, k=20)

### 3.3 Compared with random networks

In [ ]:
na.compare_with_random(G, n_samples=10)

### 3.4 Who holds pop together? Centrality

In [ ]:
na.top_centrality(G, k=10)

### 3.5 The families of pop: communities (Q2)

In [ ]:
partition, Q = na.communities(G)
null = na.modularity_null(G, n_samples=50)
print(f"Modularity {Q:.3f} vs random {null.mean():.3f} ± {null.std():.3f}")
pd.Series(partition).value_counts().head(10)

In [ ]:
# TODO: add node attributes (debut year, country, gender) then:
# na.assortativity(G, "debut_year")
na.interactive_html(G, partition, config.ROOT / "website" / "network.html")

### 3.6 Text: words that define each community (Q3)

In [ ]:
df["community"] = df["artist"].map(lambda a: partition.get(f"A:{a}"))
top_words = ta.tfidf_by_group(df.dropna(subset=["community"]).groupby("community")["tokens"].sum().to_dict())
for c in list(top_words)[:6]:
    print(c, [w for w, _ in top_words[c][:10]])
    ta.wordcloud_for(dict(top_words[c]), config.FIGURES / f"wordcloud_{c}.png")

### 3.7 Sentiment: how does each community feel?

In [ ]:
labmt = ta.load_labmt()
df["happiness"] = df["tokens"].map(lambda t: ta.happiness(t, labmt))
display(df.groupby("community")["happiness"].agg(["mean", "std", "count"]).head(10))
df.groupby("year")["happiness"].mean().plot(title="Average lyric happiness by year"); plt.show()

### 3.8 Network + text: do linked artists sound alike? (Q4)

In [ ]:
S = combine.artist_similarity(df)
pairs = combine.edge_vs_nonedge(G, S)
display(pairs.groupby("linked")["sim"].describe())
res = combine.permutation_test(G, S, n_perm=1000)
plt.hist(res["null"], bins=40); plt.axvline(res["observed"], color="red")
plt.title(f"Linked-pair similarity vs shuffled (p = {res['p_value']:.3f})"); plt.show()

## 4. Discussion
- **What went well:** TODO
- **What is missing / could be better:** TODO (e.g. missing credits for some songs, only top-N songs per artist, English-only lyrics)
- **Next steps:** TODO

## 5. Contributions
TODO: who did what (if working in a group).